# Classification
Predict whether a reviewer recommends a game based on behavioral features.

## Approach
* Logistic regression as interpretable baseline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_parquet('../data/steam_reviews_sampled.parquet')
df['playtime_hours'] = df['author.playtime_at_review'] / 60


In [2]:
from sklearn.model_selection import train_test_split

features = ['playtime_hours', 'steam_purchase', 'received_for_free', 
            'written_during_early_access', 'author.num_games_owned', 
            'author.num_reviews']
df = df.dropna()

X = df[features]
y = df['recommended']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

## Logistic Regression — Baseline (No Resampling)

### Confusion Matrix
* Model predicts "recommended" for almost everything. Only 318 out of ~520K predictions are negative
* It learned that guessing "recommended" every time gives ~87.5% accuracy
* Class imbalance is the problem. 87.5% positive vs 12.5% negative, as seen in EDA
* Model ignores the minority class entirely

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train,y_train)

y_pred = model.predict(X_test)

In [4]:
from sklearn.metrics import confusion_matrix
cnf = confusion_matrix(y_test, y_pred)
print(cnf)

[[    52  65164]
 [   266 455026]]


## Addressing Class Imbalance — SMOTE
* SMOTE (Synthetic Minority Oversampling Technique) creates synthetic negative reviews by interpolating between existing ones
* Applied to training data only — test data stays untouched to simulate real-world evaluation

In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before: {y_train.value_counts().to_dict()}')
print(f'After: {y_train_res.value_counts().to_dict()}')

Before: {True: 1821166, False: 260862}
After: {True: 1821166, False: 1821166}


## Logistic Regression — After SMOTE
* Accuracy: 67%. Lower than baseline 87%, but baseline was fake (just guessing positive every time)
* Negative class: precision 0.16, recall 0.40, F1 0.23. Catches more negatives now but lots of false alarms
* Positive class: precision 0.89, recall 0.71, F1 0.79. Decent but misclassifying 29% of positives as negative
* Logistic regression struggles here. Weak individual features and non-linear playtime relationship can't be captured by a linear model
* Justifies moving to tree-based model

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_res, y_train_res)
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))

[[ 25810  39406]
 [133205 322087]]


In [7]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.16      0.40      0.23     65216
        True       0.89      0.71      0.79    455292

    accuracy                           0.67    520508
   macro avg       0.53      0.55      0.51    520508
weighted avg       0.80      0.67      0.72    520508



## Random Forest
* Accuracy: 71%, positive F1: 0.82, negative F1: 0.22
* Slight improvement over logistic regression but negative class still weak
* Feature importances: playtime_hours (0.67), num_games_owned (0.22), num_reviews (0.07), everything else ~0
* Purchase context features (free, early access, steam purchase) contribute almost nothing to prediction

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_res, y_train_res)

y_pred = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 21869  43347]
 [109111 346181]]
              precision    recall  f1-score   support

       False       0.17      0.34      0.22     65216
        True       0.89      0.76      0.82    455292

    accuracy                           0.71    520508
   macro avg       0.53      0.55      0.52    520508
weighted avg       0.80      0.71      0.74    520508



In [9]:
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(feat_imp)

playtime_hours                0.67
author.num_games_owned        0.22
author.num_reviews            0.07
steam_purchase                0.02
written_during_early_access   0.02
received_for_free             0.00
dtype: float64


In [13]:
from sklearn.model_selection import cross_val_score
sample_idx = np.random.choice(len(X_train_res), 500000, replace=False)
X_sample = X_train_res.iloc[sample_idx]
y_sample = y_train_res.iloc[sample_idx]

scores = cross_val_score(rf, X_sample, y_sample, cv=5, scoring='f1_macro')
print(f'F1 scores: {scores}')
print(f'Mean F1: {scores.mean():.4f} (+/- {scores.std():.4f})')

F1 scores: [0.67237209 0.67368662 0.67239581 0.67266051 0.67511438]
Mean F1: 0.6732 (+/- 0.0011)
